[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yassermakram/tiny-llm-education/blob/main/stage1_a_char_reference.ipynb)

# Stage 1a: Zero-Layer Transformer (Character-Level) 🏮
## target: Learn statistical transition matrices (Bigrams) from scratch using PyTorch!

Welcome to the first session of the **Fanous LLM** educational track!
Today, we are stripping down language models to their absolute bare metal: **Zero Layers**. 

No Attention heads, no MLP layers, no LayerNorms. Just an **Embedding Matrix ($W_E$)** and an **Unembedding Matrix ($W_U$)** trying their best to predict the next character. 

### 💡 The Core Intuition:
A zero-layer network is mathematically identical to a **bigram table**. It only has a memory of *one* character. 
If the current character is **'ب'**, what's the most likely next character? 
- In **Formal Arabic (MSA)**: maybe **'ا'** (باسم) or **'ع'** (بعد).
- In **Egyptian Arabic (Masri)**: it's highly likely to be **'ي'** (بيعمل، بيكتب) because of the Egyptian present continuous prefix!

Let's train a model from scratch in PyTorch to see if it discovers these dialect-specific secrets!


In [ ]:
# 🛠️ Setup: Install dependencies if running on Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q torch plotly pandas datasets


In [ ]:
# 📦 Fetching MSA and Masri text from Wikipedia (Streaming!)
from datasets import load_dataset
import re

print("Starting to stream Wikipedia articles...")
# We use streaming=True so we don't have to download the whole gigabytes of Wikipedia!
msa_stream = load_dataset("wikimedia/wikipedia", "20231101.ar", split="train", streaming=True)
masri_stream = load_dataset("wikimedia/wikipedia", "20231101.arz", split="train", streaming=True)

def clean_and_collect(stream, max_chars=100000):
    text = ""
    for article in stream:
        # Simple cleanup
        cleaned = re.sub(r'\s+', ' ', article['text'])
        # Keep only Arabic characters, spaces, and essential punctuation
        cleaned = re.sub(r'[^\s\u0621-\u064A.،؟!]', '', cleaned)
        text += cleaned + " "
        if len(text) >= max_chars:
            break
    return text[:max_chars]

print("Collecting MSA...")
msa_text = clean_and_collect(msa_stream)
print("Collecting Masri...")
masri_text = clean_and_collect(masri_stream)
print(f"Loaded {len(msa_text)} chars of MSA and {len(masri_text)} chars of Masri.")
print("Masri preview:", masri_text[:150])


In [ ]:
# 🔠 Build Vocabulary
vocab = sorted(list(set(msa_text + masri_text)))
char_to_ix = {ch: i for i, ch in enumerate(vocab)}
ix_to_char = {i: ch for i, ch in enumerate(vocab)}
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size} characters.")


In [ ]:
# 📊 Create Bigram Dataset Tensors
import torch

def make_bigram_dataset(text, char_to_ix):
    inputs, targets = [], []
    for i in range(len(text) - 1):
        inputs.append(char_to_ix[text[i]])
        targets.append(char_to_ix[text[i+1]])
    return torch.tensor(inputs), torch.tensor(targets)

msa_inputs, msa_targets = make_bigram_dataset(msa_text, char_to_ix)
masri_inputs, masri_targets = make_bigram_dataset(masri_text, char_to_ix)


## 🧠 The Architecture: ZeroLayerTransformer

Our model has exactly two parameter matrices:
1. `embedding.weight` ($W_E$): Mapping character index to a vector of size `embed_dim`.
2. `unembedding.weight` ($W_U$): Mapping `embed_dim` back to vocabulary logits (no bias).

The prediction is:
$$\text{Logits} = W_U W_E[x]$$
Where $x$ is the input character index.


In [ ]:
# 💻 Define the PyTorch Zero-Layer Model
import torch.nn as nn

class ZeroLayerTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        # W_E matrix: [vocab_size, embed_dim]
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # W_U matrix: [vocab_size, embed_dim] (Linear without bias)
        self.unembedding = nn.Linear(embed_dim, vocab_size, bias=False)

    def forward(self, x):
        # x: [batch_size]
        embeds = self.embedding(x) # [batch_size, embed_dim]
        logits = self.unembedding(embeds) # [batch_size, vocab_size]
        return logits


## 📈 The Theoretical Limit (Shannon Entropy)
Before training, can we know what the absolute best possible loss is? Yes!
Since the model is just a bigram look-up table, the optimal weights will exactly represent the dataset's bigram log-probabilities. 
The minimum cross-entropy loss is mathematically bounded by the **Conditional Entropy** of the bigrams:
$$H(Y|X) = - \sum_{x,y} P(x, y) \log_2 P(y|x) \text{ bits} = - \sum_{x,y} P(x, y) \ln P(y|x) \text{ nats}$$
Let's compute this theoretical limit directly from counts!


In [ ]:
# 🧮 Compute the Theoretical Loss Floor
import torch.nn.functional as F

def compute_loss_floor(inputs, targets, vocab_size):
    # Count transitions
    counts = torch.zeros(vocab_size, vocab_size)
    for inp, tar in zip(inputs, targets):
        counts[inp, tar] += 1
    
    # Probabilities
    row_sums = counts.sum(dim=1, keepdim=True)
    # Avoid division by zero
    probs = counts / (row_sums + 1e-9)
    
    # Calculate Cross Entropy Loss: -log P(target | input) averaged over the dataset
    total_loss = 0.0
    valid_count = 0
    for inp, tar in zip(inputs, targets):
        p = probs[inp, tar]
        if p > 0:
            total_loss += -torch.log(p)
            valid_count += 1
            
    return (total_loss / valid_count).item()

msa_floor = compute_loss_floor(msa_inputs, msa_targets, vocab_size)
masri_floor = compute_loss_floor(masri_inputs, masri_targets, vocab_size)
print(f"Theoretical best possible Cross-Entropy loss:")
print(f" - MSA:   {msa_floor:.4f}")
print(f" - Masri: {masri_floor:.4f}")


In [ ]:
# 🚀 Live Training Loop
def train_model(inputs, targets, floor_loss, epochs=8, lr=0.05, embed_dim=32):
    model = ZeroLayerTransformer(vocab_size, embed_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    
    # DataLoader
    dataset = torch.utils.data.TensorDataset(inputs, targets)
    loader = torch.utils.data.DataLoader(dataset, batch_size=4096, shuffle=True)
    
    print(f"Training... (Theoretical floor is {floor_loss:.4f})")
    for epoch in range(epochs):
        epoch_loss = 0.0
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = loss_fn(logits, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch_x)
        
        avg_loss = epoch_loss / len(inputs)
        print(f"Epoch {epoch+1:02d} | Loss: {avg_loss:.4f}")
    return model

print("=== Training MSA Model ===")
msa_model = train_model(msa_inputs, msa_targets, msa_floor)
print("\n=== Training Masri Model ===")
masri_model = train_model(masri_inputs, masri_targets, masri_floor)


## 🎨 Visualizing the W_U W_E Matrix
Let's see what the network actually learned!
If we compute $W_{logits} = W_U W_E$ (shape `[vocab_size, vocab_size]`), we get the transition logits from character $i$ to character $j$.
Applying softmax over columns gives us the predicted transition probability matrix. Let's plot it as an interactive Plotly heatmap!


In [ ]:
# 📊 Render the dialect transition heatmaps
import plotly.graph_objects as go
import pandas as pd
import numpy as np

def plot_transition_heatmap(model, title):
    # Extract weights
    W_E = model.embedding.weight.data # [vocab_size, embed_dim]
    W_U = model.unembedding.weight.data # [vocab_size, embed_dim]
    
    # Logits matrix W_U * W_E^T
    logits = torch.matmul(W_U, W_E.t()) # [vocab_size, vocab_size]
    # Softmax along columns
    probs = torch.softmax(logits, dim=1).numpy()
    
    # We will filter out rare characters (like english letters or rare symbols) to keep it clean
    char_counts = np.bincount(msa_inputs.numpy()) + np.bincount(masri_inputs.numpy())
    top_indices = np.argsort(char_counts)[::-1][:30]
    top_chars = [ix_to_char[i] for i in top_indices]
    
    filtered_probs = probs[top_indices][:, top_indices]
    
    fig = go.Figure(data=go.Heatmap(
        z=filtered_probs,
        x=top_chars,
        y=top_chars,
        colorscale='Hot',
        zmin=0.0, zmax=0.3
    ))
    fig.update_layout(
        title=title,
        xaxis_title="Next Character (Predict)",
        yaxis_title="Current Character (Input)",
        width=700, height=700
    )
    fig.show()

print("Plotting heatmaps...")
plot_transition_heatmap(msa_model, "MSA Character Bigram Network (W_U W_E)")
plot_transition_heatmap(masri_model, "Masri Character Bigram Network (W_U W_E)")
